# Next Steps – Sim→Real-Transfer verbessern

Dieses Notebook setzt die nächsten Schritte nach dem Milestone um und nutzt `project_lib.py`.

**Schritt 1 – LSF-Verbreiterung (wellenlängenabhängig).** Die simulierten Spektren werden auf die DESI-Auflösung gefaltet. DESIs Auflösungsvermögen **R = λ/FWHM ist nicht konstant** (steigt von blau nach rot), daher wird R(λ) verwendet. Vorher klassifizierte der auf scharfen Sim-Spektren trainierte RF **75/80** DESI-Spektren fälschlich als **F** (Domain Shift).

**Schritt 2 – Echte F/G/K-Labels (Scaffold).** `teff_to_class` + `evaluate_on_labeled` für eine *echte* Accuracy, sobald Catas Katalog (K=1242, G=82) oder der DESI-MWS-Teff-VAC vorliegt.

> Ausführen in der conda-Umgebung `astro-jax`. Laufzeit v. a. durch die Spektren-Generierung bestimmt (feines Gitter); bei `N=50` einige Minuten auf CPU.


## Setup


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from collections import Counter
import importlib, project_lib as pl
importlib.reload(pl)
os.makedirs("figures", exist_ok=True)
print("Gemeinsame Grille:", pl.WAVE_GRID[0], "-", pl.WAVE_GRID[-1], "Å,", len(pl.WAVE_GRID), "Punkte")

## DESI-Auflösung R(λ)
R = λ/FWHM steigt bei DESI von blau nach rot. `pl.desi_resolution(λ)` liefert die (approximierte) Kurve; die resultierende Linienbreite FWHM(λ)=λ/R(λ) ist in Å etwa flach.


In [ ]:
wg = pl.WAVE_GRID
R = pl.desi_resolution(wg)
fwhm = wg / R
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(wg, R); ax[0].set_xlabel("λ [Å]"); ax[0].set_ylabel("R = λ/FWHM"); ax[0].set_title("DESI Auflösungsvermögen R(λ)")
ax[1].plot(wg, fwhm, color="C1"); ax[1].set_xlabel("λ [Å]"); ax[1].set_ylabel("FWHM [Å]"); ax[1].set_title("LSF-Breite FWHM(λ) = λ/R(λ)")
plt.tight_layout(); plt.savefig("figures/next_desi_resolution.png", dpi=130); plt.show()
print("R:", R.min().round(0), "-", R.max().round(0), "| FWHM [Å]:", fwhm.min().round(2), "-", fwhm.max().round(2))

### Ergebnis / Result / Resultado — DESI resolution R(λ)

**EN:** R = λ/FWHM is not constant: it rises from ~2000 in the blue to ~3800 in the red. Because R grows roughly in proportion to λ, the resulting line width FWHM(λ) = λ/R(λ) stays almost flat (~1.8 Å) across the range. Using R(λ) is therefore the physically correct parametrization of the broadening, even if the numerical effect here is close to a fixed FWHM.

**ES:** R = λ/FWHM no es constante: sube de ~2000 en el azul a ~3800 en el rojo. Como R crece casi proporcional a λ, el ancho de línea resultante FWHM(λ) = λ/R(λ) queda casi plano (~1.8 Å) en todo el rango. Por eso usar R(λ) es la parametrización físicamente correcta del ensanchamiento, aunque el efecto numérico aquí sea parecido a un FWHM fijo.

## Daten & Emulator laden
Die echten DESI-Spektren sind bereits bereinigt und auf die gemeinsame Grille gebracht (`desi_sample.npz`).


In [ ]:
d = np.load("desi_sample.npz")
X_real = d["X"]
print("DESI real:", X_real.shape)

import jax; jax.config.update("jax_enable_x64", True)
import transformer_payne as tp
emu = tp.TransformerPayne.download()
print("Emulator geladen.")

## Schritt 1a · Baseline ohne Verbreiterung (reproduziert den F-Bias)


In [ ]:
N = 50  # Spektren pro Klasse

df_before = pl.build_balanced_dataset(emu, n_per_class=N, base_seed=100)
res_before = pl.train_rf(df_before)
pred_before = np.array(res_before["classes"])[res_before["model"].predict(X_real)]
dist_before = Counter(pred_before)
print("Sim-Test-Accuracy (before):", round(res_before["accuracy"], 3))
print("DESI-Vorhersage (before):  ", dict(dist_before))

## Schritt 1b · Mit wellenlängenabhängiger LSF-Verbreiterung R(λ)
`broaden_R="desi"` faltet jedes Sim-Spektrum auf die DESI-Auflösungskurve R(λ) (exakt über einen Koordinaten-Warp, eine Faltung deckt das variable R ab). Alternativen: `broaden_R=2500` (konstantes R) oder `broaden_fwhm_A=2.0` (fester FWHM).


In [ ]:
df_after = pl.build_balanced_dataset(
    emu, n_per_class=N, base_seed=100,
    broaden_R="desi",   # wellenlängenabhängig; oder =2500 (konstant) / broaden_fwhm_A=2.0
)
res_after = pl.train_rf(df_after)
pred_after = np.array(res_after["classes"])[res_after["model"].predict(X_real)]
dist_after = Counter(pred_after)
print("Sim-Test-Accuracy (after): ", round(res_after["accuracy"], 3))
print("DESI-Vorhersage (after):   ", dict(dist_after))

## Vergleich before/after
Erwartung: Der starke F-Überhang geht zurück, die Verteilung verschiebt sich Richtung K/G (Catas realer Katalog ist K-dominiert).


In [ ]:
classes = res_after["classes"]
b = [dist_before.get(c, 0) for c in classes]
a = [dist_after.get(c, 0) for c in classes]
x = np.arange(len(classes)); w = 0.38
plt.figure(figsize=(6, 4))
plt.bar(x - w/2, b, w, label="before (scharf)")
plt.bar(x + w/2, a, w, label="after (R(λ)-verbreitert)")
plt.xticks(x, classes); plt.ylabel("# DESI-Spektren")
plt.title("DESI-Vorhersageverteilung: Effekt der R(λ)-Verbreiterung")
plt.legend(); plt.tight_layout()
plt.savefig("figures/next_desi_pred_before_after.png", dpi=130); plt.show()

### Ergebnis / Result / Resultado — Effect on the DESI predictions

**EN:** Before broadening, the sim-trained RF predicts almost only F on real DESI data (F-bias ≈ 75/80) — a symptom of the sim→real domain shift. After broadening the simulated spectra to the DESI resolution, the predicted distribution should shift toward K/G, matching the K-dominated real catalog (K=1242, G=82). Read the bar chart as: the smaller the F bar and the larger K/G after broadening, the more the domain gap was closed. The simulated test accuracy stays ~1.0 either way (that task is trivial); what matters is the change on real data.

**ES:** Antes del ensanchamiento, el RF (entrenado en simulados) predice casi solo F en datos DESI reales (sesgo F ≈ 75/80) — síntoma del domain shift sim→real. Tras ensanchar los espectros simulados a la resolución DESI, la distribución predicha debería desplazarse hacia K/G, acorde con el catálogo real dominado por K (K=1242, G=82). Lee el gráfico así: cuanto menor la barra F y mayores K/G tras ensanchar, más se cerró la brecha de dominio. La accuracy de test simulada sigue ~1.0 en ambos casos (esa tarea es trivial); lo relevante es el cambio en datos reales.

### Mittleres Spektrum: nähert sich Sim durch Verbreiterung an DESI an?


In [ ]:
mean_before = np.vstack(df_before["normalized_intensity"]).mean(0)
mean_after  = np.vstack(df_after["normalized_intensity"]).mean(0)
mean_desi   = X_real.mean(0)
plt.figure(figsize=(12, 4))
plt.plot(wg, mean_before, lw=0.6, alpha=0.8, label="sim mean (before)")
plt.plot(wg, mean_after,  lw=0.9, label="sim mean (after, R(λ))")
plt.plot(wg, mean_desi,   lw=0.9, color="k", label="DESI mean (real)")
for name,(lam,_) in pl.SPECTRAL_LINES.items():
    plt.axvline(lam, color="grey", ls="--", lw=0.6, alpha=0.5)
plt.xlabel("Wellenlänge [Å]"); plt.ylabel("Norm. Intensität"); plt.legend()
plt.title("Mittleres Spektrum – Verbreiterung nähert Sim an DESI an")
plt.tight_layout(); plt.savefig("figures/next_mean_sim_vs_desi.png", dpi=130); plt.show()

### Ergebnis / Result / Resultado — Mean spectrum (domain shift)

**EN:** The broadened simulated mean (after) should sit closer to the real DESI mean than the sharp version (before): its absorption lines become shallower and wider, matching the instrumentally broadened real lines. Remaining gaps (continuum shape, noise, residual line depth) point to the next steps — e.g. matching the SNR or cross-matching real Teff labels for a true accuracy.

**ES:** La media simulada ensanchada (after) debería quedar más cerca de la media DESI real que la versión aguda (before): sus líneas de absorción se vuelven menos profundas y más anchas, acorde con las líneas reales. Las diferencias que queden (forma del continuo, ruido, profundidad residual de líneas) indican los siguientes pasos — p.ej. igualar el SNR o cruzar etiquetas reales de Teff para una accuracy verdadera.

### Übergabe-Artefakte (verbreitertes Modell)


In [ ]:
mpath, cpath = pl.export_sim2real(res_after, X_real, out_dir=".", tag="sim2real_desiR")
print("gespeichert:", mpath, cpath)

## Schritt 2 · Echte F/G/K-Labels (Scaffold)
Bis jetzt gibt es für die DESI-Sterne keine Ground-Truth – nur `spectype=STAR`. Sobald echte Labels da sind (Catas Katalog **oder** Teff aus dem DESI-MWS-VAC):

```python
teff   = ...                        # 1D-Array der Teff-Werte der DESI-Sterne
y_true = pl.teff_to_class(teff)     # -> 'F'/'G'/'K' (Grenzen 5250/6050 K)
ev = pl.evaluate_on_labeled(res_after, X_real, y_true)
print(ev['accuracy']); print(ev['report'])
```

Wichtig: `X_real` und `y_true` müssen dieselbe Sternmenge/Reihenfolge betreffen.


In [ ]:
print("teff_to_class([4200, 5400, 6200]) ->", list(pl.teff_to_class([4200, 5400, 6200])))
print("Grenzen: K < 5250 <= G < 6050 <= F")

## Hinweise
- **Verbreiterung:** `broaden_R="desi"` nutzt R(λ) (wellenlängenabhängig). Zum Tunen: konstantes `broaden_R=2500` oder fester `broaden_fwhm_A`. Die R(λ)-Tabelle in `pl.DESI_R_TABLE_*` ist approximiert – bei Bedarf mit echten DESI-Werten ersetzen. Optional zusätzlich das SNR der Sim an DESI angleichen.
- **Feature Importance (Bericht-Präzisierung):** Die wichtigsten Features liegen auf **Balmer-Linien (Hα/Hβ/Hγ) plus einem Metalllinien-Wald bei 4000–4600 Å** – nicht ausschließlich auf den Balmer-Linien (Top-Peak ~4421 Å ist ein Metalllinien-Blend).
- **Danach:** echte Teff-Labels via DESI-MWS-VAC cross-matchen, dann `evaluate_on_labeled` für die reale Accuracy.
